# 08 — Final Animation Prompt Generation

This notebook uses the processed project data to generate a final animation narrative and scene prompts for the Part 2 animation.

The output supports the final sequence:

1. Blender fragment library appears
2. Fragments dissolve into TouchDesigner point clouds
3. Alice-related point clouds break, vibrate, and switch with audio
4. A final Alice scene is generated as the narrative conclusion

In [ ]:
import os
print(os.getenv("OPENAI_API_KEY"))

In [9]:
from pathlib import Path
import os
import json
import pandas as pd

from openai import OpenAI

## 1. Path settings

This notebook reads the processed parameter data and the available Alice point cloud assets.

The generated script and prompts are saved into:

`outputs/prompts/`

In [10]:
PARAMETER_TABLE = Path("../../data/processed/parameters/parameter_table.csv")
ALICE_POINTCLOUD_DIR = Path("../../data/processed/alice_pointclouds")
AUDIO_VECTOR_DIR = Path("../../data/processed/audio_vectors")

OUTPUT_DIR = Path("../../outputs/prompts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUTPUT_DIR.resolve())

Output folder: D:\Work\Workspace\Projects\Python\data-driven-surface\outputs\prompts


## 2. Load project data

This cell loads the mapped data from Part 1 and checks the available Alice point cloud assets.

In [11]:
parameter_df = pd.read_csv(PARAMETER_TABLE)

alice_assets = sorted([
    f.stem for f in ALICE_POINTCLOUD_DIR.glob("*.csv")
])

audio_files = sorted([
    f.name for f in AUDIO_VECTOR_DIR.glob("*")
])

print("Parameter table shape:", parameter_df.shape)
print("Alice assets:", alice_assets)
print("Audio vector files:", audio_files[:10])

Parameter table shape: (50, 21)
Alice assets: ['alice', 'card', 'cat', 'clock', 'door', 'hat', 'mushroom', 'rabbit', 'tea']
Audio vector files: []


## 3. Prepare project summary for prompt generation

The summary explains how the datasets connect to the final animation.

In [12]:
project_summary = {
    "theme": "Alice in Wonderland as a data-driven dreamscape",
    "completed_stages": {
        "stage_1": "Blender fragment asset library has already been created.",
        "stage_2": "TouchDesigner point cloud system has already been created using 9 Alice-related 3D models and audio-driven point vibration/switching."
    },
    "stage_3_goal": "Generate the final AI-assisted Alice scene animation prompt, used as the closing sequence after the Blender and TouchDesigner parts.",
    "alice_elements": [
        "rabbit", "alice", "cheshire cat", "mad hatter hat", "tea set",
        "playing card soldier", "pocket watch", "door", "mushroom"
    ],
    "audio_logic": "The previous TD stage uses the audio dataset to drive point cloud vibration, fragmentation and switching rhythm."
}

print(json.dumps(project_summary, indent=2))

{
  "theme": "Alice in Wonderland as a data-driven dreamscape",
  "completed_stages": {
    "stage_1": "Blender fragment asset library has already been created.",
    "stage_2": "TouchDesigner point cloud system has already been created using 9 Alice-related 3D models and audio-driven point vibration/switching."
  },
  "stage_3_goal": "Generate the final AI-assisted Alice scene animation prompt, used as the closing sequence after the Blender and TouchDesigner parts.",
  "alice_elements": [
    "rabbit",
    "alice",
    "cheshire cat",
    "mad hatter hat",
    "tea set",
    "playing card soldier",
    "pocket watch",
    "door",
    "mushroom"
  ],
  "audio_logic": "The previous TD stage uses the audio dataset to drive point cloud vibration, fragmentation and switching rhythm."
}


## 4. OpenAI API setup

The API key should be stored as an environment variable:

`OPENAI_API_KEY`

The notebook uses the OpenAI Python SDK to generate structured text outputs. See the official OpenAI API documentation for setup and usage. 

In [13]:
client = OpenAI()

MODEL_NAME = "gpt-4.1-mini"

## 5. Generate one-minute animation narrative

This prompt asks the model to produce a clear 60–75 second animation structure.

In [14]:
user_prompt = f"""
Create only the third-stage closing animation narrative for this project.

Context:
{json.dumps(project_summary, indent=2)}

Important:
- Do NOT describe the Blender fragment stage.
- Do NOT describe the TouchDesigner point cloud stage.
- Assume both have already happened before this scene.
- This is the final AI-assisted Alice scene sequence that follows them.
- It should feel like the point clouds have resolved into a complete Wonderland scene.
- Include all 9 Alice elements.
- The sequence should last around 20–30 seconds, as the closing part of a longer >1 minute animation.
- Output as a timestamped structure.
- Use British English.
"""

response = client.responses.create(
    model=MODEL_NAME,
    input=[
        {"role": "system", "content": "You write concise cinematic animation structures for design portfolio films."},
        {"role": "user", "content": user_prompt}
    ]
)

animation_narrative = response.output_text
print(animation_narrative)

00:00–00:05  
The camera gently zooms out as the fragmented point clouds begin converging and solidifying, seamlessly assembling into a vibrant, surreal Wonderland glade bathed in soft, dappled light.

00:05–00:10  
The rabbit, rendered in smooth AI-generated detail, looks up and darts across the scene, leading the eye towards Alice standing poised beside a giant mushroom, her features animated with subtle, dreamlike clarity.

00:10–00:15  
The Cheshire Cat materialises on a twisting branch above, its mischievous grin forming and fading, while the Mad Hatter’s iconic hat spins slowly in midair beside a whimsical tea set arranged on a floating table.

00:15–00:20  
A pocket watch hangs suspended, its hands ticking forward in rhythmic sync with the ambient soundtrack; nearby, playing card soldiers march steadily toward an ornate door that creaks open to reveal a glowing, inviting passage.

00:20–00:25  
All elements pulse softly with an underlying audio-driven vibration, echoing the prio

## 6. Generate MidJourney scene prompts

This cell generates prompts for final scene images or animation references.

In [15]:
mj_prompt_request = f"""
Generate 5 MidJourney prompts for the final closing Alice in Wonderland scene only.

Context:
{json.dumps(project_summary, indent=2)}

Requirements:
- This is only the final scene after Blender and TouchDesigner have already appeared.
- Include all 9 Alice elements if possible.
- Make it suitable for a stylised 3D animation reference.
- The scene should feel like a complete Wonderland environment formed from previous data fragments.
- Avoid clutter.
- Use clear spatial composition.
- Use cinematic lighting.
- Include MidJourney parameters: --v 6 --style raw --ar 16:9

Return only the prompts, numbered.
"""

response = client.responses.create(
    model=MODEL_NAME,
    input=[
        {"role": "system", "content": "You write clean MidJourney prompts for stylised 3D animation scenes."},
        {"role": "user", "content": mj_prompt_request}
    ]
)

midjourney_prompts = response.output_text
print(midjourney_prompts)

1. A stylised 3D Wonderland dreamscape at twilight, Alice stands at the center holding a glowing pocket watch, surrounded by floating fragmented data shards forming a large ornate door behind her; the rabbit hops nearby, playing card soldiers march in rhythmic formation, the Cheshire Cat’s smiling face emerges subtly from twisting vibrant mushrooms, and the Mad Hatter’s hat floats gently above a twinkling tea set on a reflective surface, cinematic soft blue and purple lighting, clear spatial depth, minimalist but immersive composition --v 6 --style raw --ar 16:9

2. Final Alice in Wonderland scene in stylised 3D, Alice seated calmly on a giant mushroom composed of luminous data fragments, the Mad Hatter’s iconic hat hovers overhead, the rabbit peeks from behind a semi-transparent digital door, Cheshire Cat’s eyes glowing in the distance, pocket watch suspended mid-air ticking softly, tea set arranged neatly beside card soldier figures standing guard, ambient warm cinematic lighting wit

## 7. Generate portfolio workflow explanation

This text can be adapted for the final PDF.

In [16]:
portfolio_text_request = f"""
Write a concise portfolio explanation for the third-stage final AI-assisted scene only.

Context:
{json.dumps(project_summary, indent=2)}

Requirements:
- Do not explain the Blender or TouchDesigner stages in detail.
- Mention that this final scene follows the completed Blender fragment library and audio-reactive TouchDesigner point cloud transformation.
- Explain how the final scene uses GPT-generated prompts to compose the 9 Alice elements into a concluding Wonderland environment.
- Use British English.
- Keep it around 120–160 words.
"""

response = client.responses.create(
    model=MODEL_NAME,
    input=[
        {"role": "system", "content": "You write clear portfolio text for computational design projects."},
        {"role": "user", "content": portfolio_text_request}
    ]
)

portfolio_explanation = response.output_text
print(portfolio_explanation)

The final stage presents an AI-assisted scene animation that concludes the data-driven Alice in Wonderland experience, following the completed Blender fragment asset library and the audio-reactive TouchDesigner point cloud transformations. Leveraging GPT-generated prompts, this closing sequence artfully interweaves the nine key Alice elements—rabbit, Alice, Cheshire Cat, Mad Hatter’s hat, tea set, playing card soldier, pocket watch, door, and mushroom—into a seamless, evocative Wonderland environment. The AI-driven prompt synthesis directs the dynamic composition and fluid interplay of these components, creating a richly layered dreamscape that both honours and reimagines the story’s surreal essence. This final animation encapsulates the project’s fusion of computational design, generative language models, and audio-responsive visual storytelling, offering a compelling, immersive finale to the narrative journey.


## 8. Save outputs

In [17]:
outputs = {
    "animation_narrative": animation_narrative,
    "midjourney_prompts": midjourney_prompts,
    "portfolio_explanation": portfolio_explanation
}

with open(OUTPUT_DIR / "final_animation_prompt_outputs.json", "w", encoding="utf-8") as f:
    json.dump(outputs, f, indent=2, ensure_ascii=False)

with open(OUTPUT_DIR / "final_animation_narrative.txt", "w", encoding="utf-8") as f:
    f.write(animation_narrative)

with open(OUTPUT_DIR / "midjourney_final_scene_prompts.txt", "w", encoding="utf-8") as f:
    f.write(midjourney_prompts)

with open(OUTPUT_DIR / "portfolio_workflow_explanation.txt", "w", encoding="utf-8") as f:
    f.write(portfolio_explanation)

print("Saved outputs to:", OUTPUT_DIR.resolve())

Saved outputs to: D:\Work\Workspace\Projects\Python\data-driven-surface\outputs\prompts
